# Project Lab: Five Math-from-Scratch Projects

Notebook for **Chapter 17** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch17.html

Tip: change a number, predict the output, then run the cell again.

## 17.1 How to run a math-from-scratch project

## 17.2 Project 1: PCA from scratch on a customer dataset

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 500
names = ["monthly_spend", "orders", "avg_basket", "discount_use", "site_visits",
         "session_min", "app_opens", "tenure_months", "account_days", "support_tickets"]
# three hidden traits: value (0), engagement (1), loyalty (2)
L = np.array([[0.9, 0.1, 0.0], [0.7, 0.4, 0.0], [0.8, -0.3, 0.0], [-0.3, 0.2, 0.1],
              [0.1, 0.9, 0.0], [0.0, 0.8, 0.1], [0.1, 0.85, -0.1], [0.0, 0.1, 0.95],
              [0.0, 0.1, 0.95], [0.1, 0.2, 0.4]])
F = rng.normal(size=(n, 3))
X = F @ L.T + 0.4 * rng.normal(size=(n, 10))
units = np.array([120, 8, 35, 0.2, 20, 15, 30, 24, 730, 2])      # make raw units differ
X = 50 + X * units                                               # raw customer table

print("shape:", X.shape)
print("raw std of first 4 columns:", np.round(X.std(axis=0, ddof=1)[:4], 1))
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)                 # standardise
print("Z mean ~ 0:", np.allclose(Z.mean(axis=0), 0), "| Z std = 1:", np.allclose(Z.std(axis=0, ddof=1), 1))
R = np.corrcoef(X, rowvar=False)
print("corr(tenure_months, account_days) =", round(R[7, 8], 3))
print("corr(monthly_spend, orders)       =", round(R[0, 1], 3))
print("corr(monthly_spend, site_visits)  =", round(R[0, 4], 3))

shape: (500, 10)
raw std of first 4 columns: [1.178e+02 6.900e+00 3.250e+01 1.000e-01]
Z mean ~ 0: True | Z std = 1: True
corr(tenure_months, account_days) = 0.835
corr(monthly_spend, orders)       = 0.752
corr(monthly_spend, site_visits)  = 0.117


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 500
names = ["monthly_spend", "orders", "avg_basket", "discount_use", "site_visits",
         "session_min", "app_opens", "tenure_months", "account_days", "support_tickets"]
L = np.array([[0.9, 0.1, 0.0], [0.7, 0.4, 0.0], [0.8, -0.3, 0.0], [-0.3, 0.2, 0.1],
              [0.1, 0.9, 0.0], [0.0, 0.8, 0.1], [0.1, 0.85, -0.1], [0.0, 0.1, 0.95],
              [0.0, 0.1, 0.95], [0.1, 0.2, 0.4]])
F = rng.normal(size=(n, 3))
X = 50 + (F @ L.T + 0.4 * rng.normal(size=(n, 10))) * np.array([120, 8, 35, 0.2, 20, 15, 30, 24, 730, 2])
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

C = Z.T @ Z / (n - 1)                       # covariance of standardised data = correlation matrix
vals, vecs = np.linalg.eigh(C)              # eigh: for symmetric matrices, ascending order
order = np.argsort(vals)[::-1]              # largest first
vals, vecs = vals[order], vecs[:, order]

ratio = vals / vals.sum()
print("eigenvalues      :", np.round(vals, 3))
print("sum of eigenvalues:", round(vals.sum(), 6), "(= number of features)")
print("explained ratio  :", np.round(ratio, 3))
print("cumulative       :", np.round(np.cumsum(ratio), 3))
print("components with eigenvalue > 1 (Kaiser):", int((vals > 1).sum()))

print("\nloadings (columns = PC1, PC2, PC3)")
for name, row in zip(names, vecs[:, :3]):
    print(f"{name:15s}", np.round(row, 2))
print("\northonormal columns:", np.allclose(vecs.T @ vecs, np.eye(10)))

eigenvalues      : [3.485 2.693 2.115 0.5   0.314 0.218 0.186 0.174 0.165 0.151]
sum of eigenvalues: 10.0 (= number of features)
explained ratio  : [0.348 0.269 0.212 0.05  0.031 0.022 0.019 0.017 0.017 0.015]
cumulative       : [0.348 0.618 0.829 0.879 0.911 0.932 0.951 0.968 0.985 1.   ]
components with eigenvalue > 1 (Kaiser): 3

loadings (columns = PC1, PC2, PC3)
monthly_spend   [-0.14  0.55  0.01]
orders          [-0.29  0.43  0.12]
avg_basket      [ 0.05  0.55 -0.11]
discount_use    [-0.18 -0.43  0.06]
site_visits     [-0.44 -0.06  0.28]
session_min     [-0.45 -0.1   0.22]
app_opens       [-0.41 -0.01  0.36]
tenure_months   [-0.26 -0.07 -0.55]
account_days    [-0.28 -0.07 -0.53]
support_tickets [-0.38  0.03 -0.37]

orthonormal columns: True


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 500
names = ["monthly_spend", "orders", "avg_basket", "discount_use", "site_visits",
         "session_min", "app_opens", "tenure_months", "account_days", "support_tickets"]
L = np.array([[0.9, 0.1, 0.0], [0.7, 0.4, 0.0], [0.8, -0.3, 0.0], [-0.3, 0.2, 0.1],
              [0.1, 0.9, 0.0], [0.0, 0.8, 0.1], [0.1, 0.85, -0.1], [0.0, 0.1, 0.95],
              [0.0, 0.1, 0.95], [0.1, 0.2, 0.4]])
F = rng.normal(size=(n, 3))
X = 50 + (F @ L.T + 0.4 * rng.normal(size=(n, 10))) * np.array([120, 8, 35, 0.2, 20, 15, 30, 24, 730, 2])
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
from sklearn.decomposition import PCA

# --- PCA by SVD of the standardised data matrix ---
U, S, Vt = np.linalg.svd(Z, full_matrices=False)
eig_from_svd = S**2 / (n - 1)                       # eigenvalues of the covariance matrix
C = Z.T @ Z / (n - 1)
eig_from_eigh = np.sort(np.linalg.eigvalsh(C))[::-1]
print("max |SVD eig - eigh eig| : below 1e-10:", np.abs(eig_from_svd - eig_from_eigh).max() < 1e-10)

scores = Z @ Vt.T                                    # coordinates in the new axes
print("variance of scores = eig :", np.allclose(scores.var(axis=0, ddof=1), eig_from_svd))
print("scores uncorrelated      :", np.allclose(np.corrcoef(scores, rowvar=False), np.eye(10), atol=1e-8))

# --- reconstruction with k components ---
print("\n k  rel. error (direct)  rel. error (from S)  variance kept")
for k in range(1, 7):
    Zk = scores[:, :k] @ Vt[:k]
    direct = np.linalg.norm(Z - Zk) / np.linalg.norm(Z)
    from_s = np.sqrt((S[k:] ** 2).sum() / (S**2).sum())
    print(f"{k:2d}  {direct:17.4f}  {from_s:19.4f}  {1 - from_s**2:13.3f}")

# --- compare with scikit-learn ---
sk = PCA(n_components=10).fit(Z)
print("\nmax |ratio diff| vs sklearn: below 1e-10:", np.abs(sk.explained_variance_ratio_ - eig_from_svd / eig_from_svd.sum()).max() < 1e-10)
print("max |abs(components) diff| : below 1e-10:", np.abs(np.abs(sk.components_) - np.abs(Vt)).max() < 1e-10)

max |SVD eig - eigh eig| : below 1e-10: True
variance of scores = eig : True
scores uncorrelated      : True

 k  rel. error (direct)  rel. error (from S)  variance kept
 1             0.8072               0.8072          0.348
 2             0.6183               0.6183          0.618
 3             0.4132               0.4132          0.829
 4             0.3474               0.3474          0.879
 5             0.2989               0.2989          0.911
 6             0.2600               0.2600          0.932

max |ratio diff| vs sklearn: below 1e-10: True
max |abs(components) diff| : below 1e-10: True


## 17.3 Project 2: Linear and ridge regression, three ways

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
n_train, n_test, p = 25, 500, 8
shared = rng.normal(size=(n_train + n_test, 1))
X = 0.9 * shared + 0.45 * rng.normal(size=(n_train + n_test, p))     # correlated features (r about 0.8)
beta_true = np.array([3.0, -2.0, 0.0, 0.0, 1.5, 0.0, 0.0, -1.0])
y = 5.0 + X @ beta_true + 3.0 * rng.normal(size=n_train + n_test)
Xtr, ytr, Xte, yte = X[:n_train], y[:n_train], X[n_train:], y[n_train:]
from scipy.linalg import solve_triangular
from sklearn.linear_model import LinearRegression

A = np.column_stack([np.ones(n_train), Xtr])          # design matrix with intercept column

# 1) normal equation: (A^T A) w = A^T y
w_ne = np.linalg.solve(A.T @ A, A.T @ ytr)
# 2) QR: A = QR, then R w = Q^T y (back substitution)
Q, R = np.linalg.qr(A)
w_qr = solve_triangular(R, Q.T @ ytr)
# 3) SVD-based least squares
w_ls = np.linalg.lstsq(A, ytr, rcond=None)[0]

print("intercept + coefficients (normal eq):", np.round(w_ne, 3))
print("max diff QR vs normal : below 1e-10:", np.abs(w_qr - w_ne).max() < 1e-10)
print("max diff lstsq vs QR  : below 1e-10:", np.abs(w_ls - w_qr).max() < 1e-10)

sk = LinearRegression().fit(Xtr, ytr)
print("max diff vs sklearn   : below 1e-10:", np.abs(np.r_[sk.intercept_, sk.coef_] - w_ne).max() < 1e-10)

print("\ncond(A)      =", round(np.linalg.cond(A), 2))
print("cond(A^T A)  =", round(np.linalg.cond(A.T @ A), 2), "(about cond(A) squared)")

rmse = lambda w, Xm, ym: np.sqrt(np.mean((np.column_stack([np.ones(len(Xm)), Xm]) @ w - ym) ** 2))
print("\ntrain RMSE:", round(rmse(w_ls, Xtr, ytr), 3), "| test RMSE:", round(rmse(w_ls, Xte, yte), 3))

# a nastier case: two almost identical columns
rng2 = np.random.default_rng(0)
a = rng2.normal(size=200)
B = np.column_stack([a, a + 1e-7 * rng2.normal(size=200)])
w_true = np.array([1.0, 2.0])
yb = B @ w_true
w_n = np.linalg.solve(B.T @ B, B.T @ yb)
Qb, Rb = np.linalg.qr(B)
w_q = solve_triangular(Rb, Qb.T @ yb)
print("\nnearly collinear: cond(B) > 1e6:", np.linalg.cond(B) > 1e6)
print("normal-equation error larger than QR error:", np.abs(w_n - w_true).max() > np.abs(w_q - w_true).max())

intercept + coefficients (normal eq): [ 5.401  3.976  0.06   0.943 -1.376 -0.499  2.338  0.027 -2.805]
max diff QR vs normal : below 1e-10: True
max diff lstsq vs QR  : below 1e-10: True
max diff vs sklearn   : below 1e-10: True

cond(A)      = 9.83
cond(A^T A)  = 96.65 (about cond(A) squared)

train RMSE: 3.003 | test RMSE: 3.753

nearly collinear: cond(B) > 1e6: True
normal-equation error larger than QR error: True


In [ ]:
import numpy as np

rng = np.random.default_rng(11)
n_train, n_test, p = 25, 500, 8
shared = rng.normal(size=(n_train + n_test, 1))
X = 0.9 * shared + 0.45 * rng.normal(size=(n_train + n_test, p))     # correlated features (r about 0.8)
beta_true = np.array([3.0, -2.0, 0.0, 0.0, 1.5, 0.0, 0.0, -1.0])
y = 5.0 + X @ beta_true + 3.0 * rng.normal(size=n_train + n_test)
Xtr, ytr, Xte, yte = X[:n_train], y[:n_train], X[n_train:], y[n_train:]

mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
Xs = (Xtr - mu) / sd                                  # standardised, so no intercept needed after centring y
yc = ytr - ytr.mean()
w_exact = np.linalg.lstsq(Xs, yc, rcond=None)[0]

H = Xs.T @ Xs / n_train                               # Hessian of (1/2n)||Xw - y||^2
eigs = np.linalg.eigvalsh(H)
L_, mu_ = eigs.max(), eigs.min()
print("largest / smallest Hessian eigenvalue:", round(L_, 3), "/", round(mu_, 3))
print("stable if lr < 2/L =", round(2 / L_, 3), "| best fixed lr = 2/(L+mu) =", round(2 / (L_ + mu_), 3))

def gd(lr, steps=5000, tol=1e-6):
    w = np.zeros(p)
    for t in range(1, steps + 1):
        grad = Xs.T @ (Xs @ w - yc) / n_train
        w = w - lr * grad
        if not np.isfinite(w).all() or np.linalg.norm(w) > 1e6:
            return "diverged", t
        if np.linalg.norm(w - w_exact) < tol:
            return "converged", t
    return "not yet", steps

print("\n lr      verdict      steps")
for lr in [0.01, 0.05, 0.2, 0.5, 2 / (L_ + mu_), 0.99 * 2 / L_, 1.01 * 2 / L_]:
    verdict, t = gd(lr)
    print(f"{lr:6.3f}  {verdict:10s}  {t:6d}")

largest / smallest Hessian eigenvalue: 6.195 / 0.077
stable if lr < 2/L = 0.323 | best fixed lr = 2/(L+mu) = 0.319

 lr      verdict      steps
 0.010  not yet       5000
 0.050  converged     2834
 0.200  converged      704
 0.500  diverged        19
 0.319  converged      547
 0.320  converged      674
 0.326  diverged       709


In [ ]:
import numpy as np

rng = np.random.default_rng(11)
n_train, n_test, p = 25, 500, 8
shared = rng.normal(size=(n_train + n_test, 1))
X = 0.9 * shared + 0.45 * rng.normal(size=(n_train + n_test, p))     # correlated features (r about 0.8)
beta_true = np.array([3.0, -2.0, 0.0, 0.0, 1.5, 0.0, 0.0, -1.0])
y = 5.0 + X @ beta_true + 3.0 * rng.normal(size=n_train + n_test)
Xtr, ytr, Xte, yte = X[:n_train], y[:n_train], X[n_train:], y[n_train:]
from sklearn.linear_model import Ridge

mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
Xs, Xt = (Xtr - mu) / sd, (Xte - mu) / sd
ym = ytr.mean()
yc = ytr - ym
U, s, Vt = np.linalg.svd(Xs, full_matrices=False)

def ridge(lam):
    """w = V diag(s / (s^2 + lam)) U^T y  -- same as (X^T X + lam I)^-1 X^T y"""
    return Vt.T @ ((s / (s**2 + lam)) * (U.T @ yc))

# check against the textbook formula and scikit-learn
lam = 10.0
w_formula = np.linalg.solve(Xs.T @ Xs + lam * np.eye(p), Xs.T @ yc)
sk = Ridge(alpha=lam).fit(Xs, ytr)
print("SVD vs formula : below 1e-10:", np.abs(ridge(lam) - w_formula).max() < 1e-10)
print("SVD vs sklearn : below 1e-10:", np.abs(ridge(lam) - sk.coef_).max() < 1e-10)

# 5-fold cross-validation on the training set only
folds = np.array_split(np.arange(n_train), 5)
def cv_rmse(lam):
    errs = []
    for f in folds:
        keep = np.setdiff1d(np.arange(n_train), f)
        Xa, ya = Xs[keep], ytr[keep]
        xm, ymn = Xa.mean(axis=0), ya.mean()
        w = np.linalg.solve((Xa - xm).T @ (Xa - xm) + lam * np.eye(p), (Xa - xm).T @ (ya - ymn))
        errs.append(np.mean((ymn + (Xs[f] - xm) @ w - ytr[f]) ** 2))
    return np.sqrt(np.mean(errs))

lams = [0.01, 1, 10, 30, 100, 300, 1000]
print("\n lambda   df    test RMSE  cv RMSE   coefficients (first 4)")
for lam in lams:
    w = ridge(lam)
    df = (s**2 / (s**2 + lam)).sum()
    test = np.sqrt(np.mean((ym + Xt @ w - yte) ** 2))
    print(f"{lam:7.2f}  {df:4.2f}  {test:8.3f}  {cv_rmse(lam):8.3f}  {np.round(w[:4], 2)}")
best = min(lams, key=cv_rmse)
print("\nlambda chosen by cross-validation:", best)

SVD vs formula : below 1e-10: True
SVD vs sklearn : below 1e-10: True

 lambda   df    test RMSE  cv RMSE   coefficients (first 4)
   0.01  7.98     3.750     4.366  [ 3.75  0.06  0.58 -1.19]
   1.00  6.77     3.571     4.105  [ 2.9   0.04  0.49 -0.73]
  10.00  3.47     3.488     3.908  [1.1  0.1  0.33 0.05]
  30.00  2.02     3.494     3.873  [0.58 0.15 0.27 0.17]
 100.00  1.02     3.452     3.924  [0.29 0.14 0.18 0.15]
 300.00  0.49     3.469     4.061  [0.14 0.09 0.1  0.09]
1000.00  0.18     3.550     4.193  [0.05 0.04 0.04 0.04]

lambda chosen by cross-validation: 30


## 17.4 Project 3: A neural network from scratch on the two-moons task

In [ ]:
import numpy as np

def make_moons(n, noise, rng):
    t = rng.uniform(0, np.pi, n // 2)
    upper = np.column_stack([np.cos(t), np.sin(t)])
    lower = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([upper, lower]) + noise * rng.normal(size=(n, 2))
    y = np.r_[np.zeros(n // 2), np.ones(n // 2)]
    return X, y

rng = np.random.default_rng(3)
X, y = make_moons(600, 0.25, rng)
idx = rng.permutation(600)
tr, te = idx[:400], idx[400:]
mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)
Xtr, ytr, Xte, yte = (X[tr] - mu) / sd, y[tr], (X[te] - mu) / sd, y[te]

print("train/test sizes:", len(Xtr), len(Xte))
print("class balance (train):", ytr.mean().round(3))
print("train mean:", Xtr.mean(axis=0).round(3), "std:", Xtr.std(axis=0).round(3))
print("first 3 points:\n", Xtr[:3].round(3), "labels", ytr[:3])

train/test sizes: 400 200
class balance (train): 0.518
train mean: [-0. -0.] std: [1. 1.]
first 3 points:
 [[-0.917  0.772]
 [-1.19   0.655]
 [-0.955  0.006]] labels [0. 0. 1.]


In [ ]:
import numpy as np

def init(sizes, rng):
    """Weights ~ N(0, 1/fan_in) (Xavier-style for tanh), biases 0."""
    return [[rng.normal(size=(a, b)) / np.sqrt(a), np.zeros(b)] for a, b in zip(sizes[:-1], sizes[1:])]

def forward(params, X):
    acts = [X]
    for W, b in params[:-1]:
        acts.append(np.tanh(acts[-1] @ W + b))           # hidden layers
    W, b = params[-1]
    logits = acts[-1] @ W + b                            # output layer: logit
    return acts, logits[:, 0]

def loss_and_grads(params, X, y):
    acts, z = forward(params, X)
    m = len(y)
    loss = np.mean(np.logaddexp(0, z) - y * z)           # stable binary cross-entropy
    dz = ((1 / (1 + np.exp(-z))) - y)[:, None] / m       # dLoss/dlogit = (p - y)/m
    grads = [None] * len(params)
    for i in range(len(params) - 1, -1, -1):
        W, b = params[i]
        grads[i] = [acts[i].T @ dz, dz.sum(axis=0)]      # dW, db
        if i > 0:
            dz = (dz @ W.T) * (1 - acts[i] ** 2)         # back through tanh: 1 - tanh^2
    return loss, grads

def flat(params):
    return np.concatenate([a.ravel() for layer in params for a in layer])

def set_flat(params, v):
    k = 0
    for layer in params:
        for a in layer:
            a[...] = v[k:k + a.size].reshape(a.shape); k += a.size

rng = np.random.default_rng(0)
X = rng.normal(size=(20, 2)); y = (rng.random(20) > 0.5).astype(float)
params = init([2, 8, 8, 1], rng)

loss, grads = loss_and_grads(params, X, y)
g_analytic = np.concatenate([a.ravel() for layer in grads for a in layer])

# numerical gradient by central differences
v0, eps = flat(params), 1e-6
g_numeric = np.zeros_like(v0)
for i in range(len(v0)):
    vp, vm = v0.copy(), v0.copy(); vp[i] += eps; vm[i] -= eps
    set_flat(params, vp); lp = loss_and_grads(params, X, y)[0]
    set_flat(params, vm); lm = loss_and_grads(params, X, y)[0]
    g_numeric[i] = (lp - lm) / (2 * eps)

rel = np.linalg.norm(g_analytic - g_numeric) / (np.linalg.norm(g_analytic) + np.linalg.norm(g_numeric))
print("parameters:", len(v0), "| initial loss:", round(loss, 4), "(ln 2 = 0.6931)")
print("log10 of relative error:", round(np.log10(rel)))
print("gradient check passed:", rel < 1e-7)

# what a bug looks like: first-layer weight gradient wrong by a factor 1.3
bad = g_analytic.copy(); bad[:16] *= 1.3
print("log10 of relative error with a wrong first-layer gradient:", round(np.log10(np.linalg.norm(bad - g_numeric) / (np.linalg.norm(bad) + np.linalg.norm(g_numeric))), 1))

parameters: 105 | initial loss: 0.6843 (ln 2 = 0.6931)
log10 of relative error: -9
gradient check passed: True
log10 of relative error with a wrong first-layer gradient: -1.5


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def make_moons(n, noise, rng):
    t = rng.uniform(0, np.pi, n // 2)
    upper = np.column_stack([np.cos(t), np.sin(t)])
    lower = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([upper, lower]) + noise * rng.normal(size=(n, 2))
    return X, np.r_[np.zeros(n // 2), np.ones(n // 2)]

rng = np.random.default_rng(3)
X, y = make_moons(600, 0.25, rng)
idx = rng.permutation(600)
tr, te = idx[:400], idx[400:]
mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)
Xtr, ytr, Xte, yte = (X[tr] - mu) / sd, y[tr], (X[te] - mu) / sd, y[te]

def init(sizes, rng):
    """Weights ~ N(0, 1/fan_in) (Xavier-style for tanh), biases 0."""
    return [[rng.normal(size=(a, b)) / np.sqrt(a), np.zeros(b)] for a, b in zip(sizes[:-1], sizes[1:])]

def forward(params, X):
    acts = [X]
    for W, b in params[:-1]:
        acts.append(np.tanh(acts[-1] @ W + b))           # hidden layers
    W, b = params[-1]
    logits = acts[-1] @ W + b                            # output layer: logit
    return acts, logits[:, 0]

def loss_and_grads(params, X, y):
    acts, z = forward(params, X)
    m = len(y)
    loss = np.mean(np.logaddexp(0, z) - y * z)           # stable binary cross-entropy
    dz = ((1 / (1 + np.exp(-z))) - y)[:, None] / m       # dLoss/dlogit = (p - y)/m
    grads = [None] * len(params)
    for i in range(len(params) - 1, -1, -1):
        W, b = params[i]
        grads[i] = [acts[i].T @ dz, dz.sum(axis=0)]      # dW, db
        if i > 0:
            dz = (dz @ W.T) * (1 - acts[i] ** 2)         # back through tanh: 1 - tanh^2
    return loss, grads

def train(opt, lr, epochs=150, batch=32, seed=1):
    rng = np.random.default_rng(seed)
    params = init([2, 16, 16, 1], rng)
    vel = [[np.zeros_like(a) for a in layer] for layer in params]     # momentum / Adam m
    sq = [[np.zeros_like(a) for a in layer] for layer in params]      # Adam v
    t, history = 0, []
    for epoch in range(1, epochs + 1):
        perm = rng.permutation(len(Xtr))
        for s in range(0, len(Xtr), batch):
            b = perm[s:s + batch]
            _, grads = loss_and_grads(params, Xtr[b], ytr[b])
            t += 1
            for i, layer in enumerate(params):
                for j, a in enumerate(layer):
                    g = grads[i][j]
                    if opt == "momentum":
                        vel[i][j] = 0.9 * vel[i][j] + g
                        a -= lr * vel[i][j]
                    else:  # adam
                        vel[i][j] = 0.9 * vel[i][j] + 0.1 * g
                        sq[i][j] = 0.999 * sq[i][j] + 0.001 * g**2
                        mh, vh = vel[i][j] / (1 - 0.9**t), sq[i][j] / (1 - 0.999**t)
                        a -= lr * mh / (np.sqrt(vh) + 1e-8)
        if epoch in (1, 10, 50, 150):
            history.append((epoch, loss_and_grads(params, Xtr, ytr)[0]))
    acc = lambda Xm, ym: np.mean((forward(params, Xm)[1] > 0) == ym)
    return history, acc(Xtr, ytr), acc(Xte, yte)

print("optimiser            loss@1  loss@10  loss@50  loss@150  train acc  test acc")
for name, opt, lr in [("SGD+momentum 0.01", "momentum", 0.01), ("SGD+momentum 0.05", "momentum", 0.05),
                      ("Adam 0.01", "adam", 0.01), ("Adam 0.05", "adam", 0.05)]:
    h, atr, ate = train(opt, lr)
    print(f"{name:19s}  " + "  ".join(f"{l:7.3f}" for _, l in h) + f"   {atr:8.3f}  {ate:8.3f}")

lr_model = LogisticRegression().fit(Xtr, ytr)
print("\nlogistic regression baseline: train", round(lr_model.score(Xtr, ytr), 3), "| test", round(lr_model.score(Xte, yte), 3))

optimiser            loss@1  loss@10  loss@50  loss@150  train acc  test acc
SGD+momentum 0.01      0.438    0.239    0.215    0.084      0.968     0.940
SGD+momentum 0.05      0.245    0.219    0.076    0.069      0.975     0.930
Adam 0.01              0.255    0.206    0.077    0.068      0.973     0.930
Adam 0.05              0.271    0.115    0.062    0.046      0.980     0.930

logistic regression baseline: train 0.892 | test 0.87


## 17.5 Project 4: PageRank and Markov-chain analysis of a 12-page web

In [ ]:
import numpy as np

labels = list("ABCDEFGHIJKL")
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 0), (3, 2), (3, 4), (4, 2), (4, 5), (5, 4), (5, 6),
         (6, 2), (6, 7), (7, 8), (8, 9), (9, 7), (9, 10), (10, 2), (10, 11), (8, 2)]   # L (11) has no out-links
n = 12
A = np.zeros((n, n))
for i, j in edges:
    A[j, i] = 1                                     # column i = where page i links to
out_deg = A.sum(axis=0)
P = np.where(out_deg > 0, A / np.where(out_deg == 0, 1, out_deg), 1 / n)   # dangling column -> uniform

print("out-degrees:", {l: int(v) for l, v in zip(labels, out_deg)})
print("dangling pages:", [labels[i] for i in np.where(out_deg == 0)[0]])
print("column sums of P all 1:", np.allclose(P.sum(axis=0), 1))
print("in-links:", {l: int(v) for l, v in zip(labels, A.sum(axis=1))})
print("P[:, 0] (page A):", P[:, 0])
print("P[:, 11] (page L, dangling):", P[:, 11].round(4))

out-degrees: {'A': 2, 'B': 2, 'C': 1, 'D': 2, 'E': 2, 'F': 2, 'G': 2, 'H': 1, 'I': 2, 'J': 2, 'K': 2, 'L': 0}
dangling pages: ['L']
column sums of P all 1: True
in-links: {'A': 1, 'B': 1, 'C': 7, 'D': 1, 'E': 2, 'F': 1, 'G': 1, 'H': 2, 'I': 1, 'J': 1, 'K': 1, 'L': 1}
P[:, 0] (page A): [0.  0.5 0.5 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
P[:, 11] (page L, dangling): [0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833
 0.0833 0.0833]


In [ ]:
import numpy as np

labels = list("ABCDEFGHIJKL")
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 0), (3, 2), (3, 4), (4, 2), (4, 5), (5, 4), (5, 6),
         (6, 2), (6, 7), (7, 8), (8, 9), (9, 7), (9, 10), (10, 2), (10, 11), (8, 2)]   # L (11) has no out-links
n = 12
A = np.zeros((n, n))
for i, j in edges:
    A[j, i] = 1                                     # column i = where page i links to
out_deg = A.sum(axis=0)
P = np.where(out_deg > 0, A / np.where(out_deg == 0, 1, out_deg), 1 / n)   # dangling column -> uniform

d = 0.85
G = d * P + (1 - d) / n * np.ones((n, n))           # Google matrix, column-stochastic

# 1) power iteration
x = np.ones(n) / n
for it in range(1, 1000):
    x_new = G @ x
    if np.abs(x_new - x).sum() < 1e-12:
        break
    x = x_new
pr_power = x_new

# 2) linear solve: x = d P x + (1-d)/n 1   =>   (I - d P) x = (1-d)/n 1
pr_solve = np.linalg.solve(np.eye(n) - d * P, (1 - d) / n * np.ones(n))

# 3) eigenvector of G for eigenvalue 1
w, V = np.linalg.eig(G)
k = np.argmin(np.abs(w - 1))
pr_eig = np.real(V[:, k]); pr_eig = pr_eig / pr_eig.sum()

print("power iteration steps:", it)
print("max diff power vs solve: below 1e-10:", np.abs(pr_power - pr_solve).max() < 1e-10)
print("max diff eig   vs solve: below 1e-10:", np.abs(pr_eig - pr_solve).max() < 1e-10)
print("sum of PageRank:", round(pr_solve.sum(), 6))
order = np.argsort(-pr_solve)
print("\nrank page  score  in-links")
for r, i in enumerate(order, 1):
    print(f"{r:4d}  {labels[i]:>3s}  {pr_solve[i]:.4f}  {int(A[i].sum()):5d}")

power iteration steps: 59
max diff power vs solve: below 1e-10: True
max diff eig   vs solve: below 1e-10: True
sum of PageRank: 1.0

rank page  score  in-links
   1    C  0.2651      7
   2    A  0.2397      1
   3    B  0.1163      1
   4    D  0.0638      1
   5    E  0.0582      2
   6    I  0.0509      1
   7    H  0.0429      2
   8    F  0.0391      1
   9    J  0.0361      1
  10    G  0.0311      1
  11    K  0.0297      1
  12    L  0.0271      1


In [ ]:
import numpy as np

labels = list("ABCDEFGHIJKL")
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 0), (3, 2), (3, 4), (4, 2), (4, 5), (5, 4), (5, 6),
         (6, 2), (6, 7), (7, 8), (8, 9), (9, 7), (9, 10), (10, 2), (10, 11), (8, 2)]   # L (11) has no out-links
n = 12
A = np.zeros((n, n))
for i, j in edges:
    A[j, i] = 1                                     # column i = where page i links to
out_deg = A.sum(axis=0)
P = np.where(out_deg > 0, A / np.where(out_deg == 0, 1, out_deg), 1 / n)   # dangling column -> uniform
from scipy.stats import kendalltau

def google(d):
    return d * P + (1 - d) / n * np.ones((n, n))

def power(d, tol=1e-10):
    G, x = google(d), np.ones(n) / n
    for it in range(1, 100000):
        xn = G @ x
        if np.abs(xn - x).sum() < tol:
            return xn, it
        x = xn

ref, _ = power(0.85)
print(" d     steps  |lambda2|  top page  top score  tau vs d=0.85")
for d in [0.5, 0.7, 0.85, 0.95, 0.99]:
    x, it = power(d)
    ev = np.sort(np.abs(np.linalg.eigvals(google(d))))[::-1]
    tau = kendalltau(x, ref)[0]
    print(f"{d:4.2f}  {it:5d}  {ev[1]:8.4f}  {labels[np.argmax(x)]:>8s}  {x.max():9.4f}  {tau:12.3f}")

# Markov chain view: simulate the random surfer
rng = np.random.default_rng(5)
d, steps, state = 0.85, 200000, 0
visits = np.zeros(n)
for _ in range(steps):
    if rng.random() < d:
        state = rng.choice(n, p=P[:, state])         # follow a link (or jump if dangling)
    else:
        state = rng.integers(n)                      # teleport
    visits[state] += 1
pi, _ = power(0.85)
print("\nmax |simulated - exact| frequency:", round(np.abs(visits / steps - pi).max(), 4))
print("expected return time to C = 1/pi_C:", round(1 / pi[2], 2), "steps")
print("expected return time to L = 1/pi_L:", round(1 / pi[11], 2), "steps")

 d     steps  |lambda2|  top page  top score  tau vs d=0.85
0.50     23    0.3706         C     0.1829         0.758
0.70     35    0.5189         C     0.2257         0.818
0.85     49    0.6301         C     0.2651         1.000
0.95     64    0.7042         C     0.2987         0.909
0.99     73    0.7339         C     0.3149         0.848

max |simulated - exact| frequency: 0.0007
expected return time to C = 1/pi_C: 3.77 steps
expected return time to L = 1/pi_L: 36.96 steps


## 17.6 Project 5: SVD image compression and a tiny recommender

In [ ]:
import numpy as np

def make_image(size=96, seed=2):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size] / (size - 1)
    img = 0.3 + 0.4 * xx * yy                                    # smooth gradient
    img += 0.25 * (((xx - 0.65) ** 2 + (yy - 0.35) ** 2) < 0.04)  # a disc
    img += 0.15 * ((np.sin(18 * xx) > 0.3) & (yy > 0.7))           # stripes in the lower band
    img += 0.2 * ((abs(xx - 0.2) < 0.07) & (yy < 0.6))             # a bar
    img += 0.02 * rng.normal(size=(size, size))                    # sensor noise
    return np.clip(img, 0, 1)

img = make_image()
m, nn = img.shape
U, S, Vt = np.linalg.svd(img, full_matrices=False)

print("image shape:", img.shape, "| pixel range:", img.min().round(2), "to", img.max().round(2))
print("top 6 singular values:", S[:6].round(2))
print("energy in first 5 / 10 / 20:", [round(float((S[:k] ** 2).sum() / (S ** 2).sum()), 4) for k in (5, 10, 20)])

print("\n k  storage  Frobenius err  Eckart-Young formula  PSNR (dB)")
for k in [1, 2, 5, 10, 20, 40]:
    approx = (U[:, :k] * S[:k]) @ Vt[:k]
    err = np.linalg.norm(img - approx)
    ey = np.sqrt((S[k:] ** 2).sum())
    psnr = 10 * np.log10(1 / np.mean((img - approx) ** 2))
    storage = k * (m + nn + 1) / (m * nn)
    print(f"{k:2d}  {storage:6.1%}  {err:13.4f}  {ey:20.4f}  {psnr:9.2f}")

image shape: (96, 96) | pixel range: 0.25 to 0.81
top 6 singular values: [45.85  5.82  3.61  1.98  1.31  1.12]
energy in first 5 / 10 / 20: [0.9968, 0.9982, 0.9989]

 k  storage  Frobenius err  Eckart-Young formula  PSNR (dB)
 1    2.1%         7.7165                7.7165      21.90
 2    4.2%         5.0684                5.0684      25.55
 5   10.5%         2.6457                2.6457      31.19
10   20.9%         1.9983                1.9983      33.63
20   41.9%         1.5207                1.5207      36.00
40   83.8%         0.9009                0.9009      40.55


In [ ]:
import numpy as np

def make_image(size=96, seed=2):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size] / (size - 1)
    img = 0.3 + 0.4 * xx * yy                                    # smooth gradient
    img += 0.25 * (((xx - 0.65) ** 2 + (yy - 0.35) ** 2) < 0.04)  # a disc
    img += 0.15 * ((np.sin(18 * xx) > 0.3) & (yy > 0.7))           # stripes in the lower band
    img += 0.2 * ((abs(xx - 0.2) < 0.07) & (yy < 0.6))             # a bar
    img += 0.02 * rng.normal(size=(size, size))                    # sensor noise
    return np.clip(img, 0, 1)
from scipy.sparse.linalg import svds

img = make_image()
U, S, Vt = np.linalg.svd(img, full_matrices=False)

# reference: ARPACK partial SVD returns the top-k triplets
k = 10
u, s, vt = svds(img, k=k)
s = s[::-1]; u = u[:, ::-1]; vt = vt[::-1]
print("max |singular value diff| vs svds: below 1e-10:", np.abs(s - S[:k]).max() < 1e-10)
print("rank-10 reconstructions agree   :", np.allclose((u * s) @ vt, (U[:, :k] * S[:k]) @ Vt[:k]))

# smallest rank for a quality target
for target in [25, 30, 35]:
    for k in range(1, 97):
        approx = (U[:, :k] * S[:k]) @ Vt[:k]
        if 10 * np.log10(1 / np.mean((img - approx) ** 2)) >= target:
            print(f"PSNR >= {target} dB needs rank {k}  (stores {k * (96 + 96 + 1) / 96**2:.1%} of the numbers)")
            break

# rank-1 building blocks: sigma_i * u_i v_i^T
first = S[0] * np.outer(U[:, 0], Vt[0])
print("\nfirst rank-1 layer alone already captures", round(1 - np.linalg.norm(img - first)**2 / np.linalg.norm(img)**2, 4), "of the energy")

max |singular value diff| vs svds: below 1e-10: True
rank-10 reconstructions agree   : True
PSNR >= 25 dB needs rank 2  (stores 4.2% of the numbers)
PSNR >= 30 dB needs rank 4  (stores 8.4% of the numbers)
PSNR >= 35 dB needs rank 16  (stores 33.5% of the numbers)

first rank-1 layer alone already captures 0.9725 of the energy


In [ ]:
import numpy as np

rng = np.random.default_rng(4)
n_users, n_items, true_rank = 120, 50, 3
Uf = rng.normal(size=(n_users, true_rank))
Vf = rng.normal(size=(n_items, true_rank))
full = np.clip(3.2 + 0.7 * (Uf @ Vf.T) + 0.3 * rng.normal(size=(n_users, n_items)), 1, 5)   # "true" ratings
mask = rng.random((n_users, n_items)) < 0.25                                          # 25% observed
test_mask = (~mask) & (rng.random((n_users, n_items)) < 0.25)                          # held-out cells
print("observed:", int(mask.sum()), "| held-out:", int(test_mask.sum()), "| density:", round(mask.mean(), 3))

def als(k, lam, iters=25, seed=0):
    r = np.random.default_rng(seed)
    mu = full[mask].mean()
    Pu, Qi = 0.1 * r.normal(size=(n_users, k)), 0.1 * r.normal(size=(n_items, k))
    R = full - mu
    for _ in range(iters):
        for u in range(n_users):                         # fix items, solve ridge for each user
            idx = mask[u]
            Qs = Qi[idx]
            Pu[u] = np.linalg.solve(Qs.T @ Qs + lam * np.eye(k), Qs.T @ R[u, idx])
        for i in range(n_items):                         # fix users, solve ridge for each item
            idx = mask[:, i]
            Ps = Pu[idx]
            Qi[i] = np.linalg.solve(Ps.T @ Ps + lam * np.eye(k), Ps.T @ R[idx, i])
    return mu + Pu @ Qi.T

rmse = lambda pred, m: np.sqrt(np.mean((pred[m] - full[m]) ** 2))
base = np.full_like(full, full[mask].mean())
print("baseline (global mean) test RMSE:", round(rmse(base, test_mask), 3))

print("\n rank  lambda  train RMSE  test RMSE")
best = None
for k in [1, 2, 3, 4, 6, 8]:
    for lam in [0.5, 5.0]:
        pred = als(k, lam)
        tr, te = rmse(pred, mask), rmse(pred, test_mask)
        if best is None or te < best[0]:
            best = (te, k, lam)
        print(f"{k:5d}  {lam:6.1f}  {tr:10.3f}  {te:9.3f}")
print("\nbest on held-out cells: rank", best[1], "lambda", best[2], "RMSE", round(best[0], 3))

observed: 1505 | held-out: 1125 | density: 0.251
baseline (global mean) test RMSE: 1.043

 rank  lambda  train RMSE  test RMSE
    1     0.5       0.777      0.855
    1     5.0       0.820      0.856
    2     0.5       0.542      0.663
    2     5.0       0.651      0.730
    3     0.5       0.267      0.410
    3     5.0       0.518      0.631
    4     0.5       0.233      0.458
    4     5.0       0.518      0.631
    6     0.5       0.173      0.518
    6     5.0       0.518      0.631
    8     0.5       0.129      0.541
    8     5.0       0.518      0.631

best on held-out cells: rank 3 lambda 0.5 RMSE 0.41


## 17.7 Portfolio checklist, presenting your projects and a four-week schedule

## Chapter recap

## Practice set

Problem 17.8.1 Two standardised features have correlation 0.6. Find the eigenvalues of the correlation matrix and the share of variance on PC1.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch17.html